#  Rule-Based Medical Diagnosis System


In [10]:
# Install dependencies
!sudo apt-get update
!sudo apt-get install -y swi-prolog
!pip install git+https://github.com/yuce/pyswip.git

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,776 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,037 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,986 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubun

In [11]:
from google.colab import files
uploaded = files.upload()


Saving grammar.pl to grammar (1).pl


In [12]:
from google.colab import files
uploaded = files.upload()


Saving kb.pl to kb (1).pl


In [13]:
with open("inference_engine.py", "w") as f:
    f.write("""
from pyswip import Prolog

class PrologInferenceEngine:
    def __init__(self, kb_path="kb.pl", grammar_path="grammar.pl"):
        self.prolog = Prolog()
        self.prolog.consult(kb_path)
        self.prolog.consult(grammar_path)

    def parse_input_to_symptoms(self, input_str):
        # Escape quotes
        input_str = input_str.replace('"', '\\"')
        query = f'parse_symptoms("{input_str}", Symptoms)'
        result = list(self.prolog.query(query))
        if result:
            return [str(sym) for sym in result[0]["Symptoms"]]
        return []

    def get_all_diseases(self):
        result = list(self.prolog.query("disease(D, _, _)"))
        return [res["D"] for res in result]

    def get_probability(self, disease, symptoms):
        prolog_list = "[" + ",".join(symptoms) + "]" if symptoms else "[]"
        query = f"get_disease_probability('{disease}', {prolog_list}, Prob)"
        try:
          result = list(self.prolog.query(query))
          if result:
              return float(result[0]["Prob"])
        except Exception as e:
              print(f"Prolog query failed for {disease}: {e}")
              return 0.0

    def rank_diseases(self, symptoms, show_debug=False):
        diseases = self.get_all_diseases()
        probabilities = []
        for d in diseases:
            p = self.get_probability(d, symptoms)
            if show_debug:
                print(f"{d}: {p:.2f}%")
            if p > 0:
                probabilities.append((d, p))
        probabilities.sort(key=lambda x: x[1], reverse=True)
        return probabilities
""")


**database.py** – symptom and disease mappings

In [14]:
with open("database.py", "w") as f:
    f.write("""
    # --- Symptom Data ---
all_symptoms = [
    "runny_or_stuffy_nose",
    "sneezing",
    "sore_throat",
    "mild_cough",
    "mild_headache",
    "low_grade_fever",
    "mild_body_aches_or_fatigue",
    "high_fever",
    "chills",
    "severe_headache",
    "muscle_aches_joint_pain",
    "dry_cough",
    "profound_fatigue_weakness",
    "nasal_congestion",
    "excessive_thirst",
    "frequent_urination",
    "increased_hunger",
    "unintentional_weight_change",
    "blurry_vision",
    "slow_healing_wounds",
    "tingling_numbness_in_extremities",
    "fatigue",
    "no_symptoms_often",
    "occasional_headaches",
    "dizziness",
    "blurred_vision",
    "nosebleeds",
    "fatigue_confusion",
    "difficulty_breathing",
    "wheezing",
    "chest_tightness",
    "coughing_worse_at_night",
    "rapid_breathing",
    "fatigue_during_attacks",
    "sneezing_fits",
    "stuffy_runny_nose_clear",
    "itchy_red_watery_eyes",
    "itchy_throat_or_nose",
    "postnasal_drip",
    "mild_fatigue",
    "persistent_sadness",
    "loss_of_interest",
    "appetite_changes",
    "significant_weight_changes",
    "sleep_disturbances",
    "fatigue_or_loss_of_energy",
    "difficulty_concentrating",
    "feelings_of_guilt",
    "thoughts_of_death_or_suicide",
    "excessive_worry",
    "restlessness",
    "irritability",
    "muscle_tension",
    "memory_loss_short_term",
    "difficulty_finding_words",
    "trouble_completing_familiar_tasks",
    "confusion_time_place",
    "poor_judgment",
    "mood_personality_changes",
    "social_withdrawal",
    "throbbing_headache_one_sided",
    "sensitivity_to_light",
    "sensitivity_to_sound",
    "nausea_vomiting",
    "visual_disturbances_auras",
    "fatigue_difficulty_concentrating",
    "neck_stiffness",
    "persistent_cough_with_mucus",
    "fever_sweating_chills",
    "shortness_of_breath",
    "rapid_shallow_breathing",
    "chest_pain_with_breathing",
    "fatigue_weakness",
    "loss_of_appetite",
    "watery_diarrhea",
    "abdominal_cramps_pain",
    "headache_body_aches",
    "dehydration_symptoms",
    "persistent_cough_3_weeks",
    "coughing_up_blood",
    "night_sweats",
    "fever_chills",
    "unintentional_weight_loss",
    "chest_pain_discomfort",
    "chronic_productive_cough",
    "shortness_of_breath_exertion",
    "wheezing_noisy_breathing",
    "frequent_respiratory_infections",
    "fatigue_reduced_exercise_tolerance",
    "joint_pain_swelling_symmetrical",
    "morning_stiffness_30mins",
    "fatigue_fever_malaise",
    "reduced_range_of_motion",
    "rheumatoid_nodules",
    "joint_deformities_long_term",
    "joint_pain_with_activity",
    "stiffness_less_30mins",
    "reduced_flexibility",
    "bone_spurs",
    "mild_joint_swelling",
    "crepitus_joints",
    "fatigue_sluggishness",
    "weight_gain",
    "cold_intolerance",
    "dry_skin_brittle_hair",
    "constipation",
    "muscle_weakness_aches",
    "depression_low_mood",
    "slow_heart_rate",
    "extreme_fatigue",
    "fever",
    "severe_sore_throat",
    "swollen_lymph_glands",
    "swollen_tonsils",
    "headache",
    "rash",
    "muscle_aches",
    "itchy_blister_like_rash",
    "fatigue_malaise",
    "rash_starts_trunk_spreads",
    "stiff_neck",
    "confusion_concentration_issues",
    "drowsiness_difficulty_waking",
    "seizures_coma",
    "common_cold" # Added common_cold
]

# --- Mapping (Internal key -> Human-readable name) ---
words_map = {
    "common_cold": "Common cold",
    "flu": "Flu",
    "pneumonia": "Pneumonia",
    "runny_or_stuffy_nose": "Runny nose or stuffy nose",
    "sneezing": "Sneezing",
    "sore_throat": "Sore throat",
    "mild_cough": "Mild cough",
    "mild_headache": "Mild headache",
    "low_grade_fever": "Low-grade fever",
    "mild_body_aches_or_fatigue": "Mild body aches or fatigue",
    "high_fever": "High fever",
    "chills": "Chills",
    "severe_headache": "Severe headache",
    "muscle_aches_joint_pain": "Muscle aches or joint pain",
    "dry_cough": "Dry cough",
    "profound_fatigue_weakness": "Profound fatigue or weakness",
    "nasal_congestion": "Nasal congestion",
    "excessive_thirst": "Excessive thirst",
    "frequent_urination": "Frequent urination",
    "increased_hunger": "Increased hunger",
    "unintentional_weight_change": "Unintentional weight change",
    "blurry_vision": "Blurry vision",
    "slow_healing_wounds": "Slow-healing wounds",
    "tingling_numbness_in_extremities": "Tingling or numbness in extremities",
    "fatigue": "Fatigue",
    "no_symptoms_often": "No symptoms often",
    "occasional_headaches": "Occasional headaches",
    "dizziness": "Dizziness",
    "blurred_vision": "Blurred vision",
    "nosebleeds": "Nosebleeds",
    "fatigue_confusion": "Fatigue or confusion",
    "difficulty_breathing": "Difficulty breathing",
    "wheezing": "Wheezing",
    "chest_tightness": "Chest tightness",
    "coughing_worse_at_night": "Coughing worse at night",
    "rapid_breathing": "Rapid breathing",
    "fatigue_during_attacks": "Fatigue during attacks",
    "sneezing_fits": "Sneezing fits",
    "stuffy_runny_nose_clear": "Stuffy or runny nose (clear)",
    "itchy_red_watery_eyes": "Itchy, red, watery eyes",
    "itchy_throat_or_nose": "Itchy throat or nose",
    "postnasal_drip": "Postnasal drip",
    "mild_fatigue": "Mild fatigue",
    "persistent_sadness": "Persistent sadness",
    "loss_of_interest": "Loss of interest",
    "appetite_changes": "Appetite changes",
    "significant_weight_changes": "Significant weight changes",
    "sleep_disturbances": "Sleep disturbances",
    "fatigue_or_loss_of_energy": "Fatigue or loss of energy",
    "difficulty_concentrating": "Difficulty concentrating",
    "feelings_of_guilt": "Feelings of guilt",
    "thoughts_of_death_or_suicide": "Thoughts of death or suicide",
    "excessive_worry": "Excessive worry",
    "restlessness": "Restlessness",
    "irritability": "Irritability",
    "muscle_tension": "Muscle tension",
    "memory_loss_short_term": "Memory loss (short term)",
    "difficulty_finding_words": "Difficulty finding words",
    "trouble_completing_familiar_tasks": "Trouble completing familiar tasks",
    "confusion_time_place": "Confusion (time/place)",
    "poor_judgment": "Poor judgment",
    "mood_personality_changes": "Mood or personality changes",
    "social_withdrawal": "Social withdrawal",
    "throbbing_headache_one_sided": "Throbbing headache (one-sided)",
    "sensitivity_to_light": "Sensitivity to light",
    "sensitivity_to_sound": "Sensitivity to sound",
    "nausea_vomiting": "Nausea or vomiting",
    "visual_disturbances_auras": "Visual disturbances or auras",
    "fatigue_difficulty_concentrating": "Fatigue or difficulty concentrating",
    "neck_stiffness": "Neck stiffness",
    "persistent_cough_with_mucus": "Persistent cough with mucus",
    "fever_sweating_chills": "Fever, sweating, chills",
    "shortness_of_breath": "Shortness of breath",
    "rapid_shallow_breathing": "Rapid, shallow breathing",
    "chest_pain_with_breathing": "Chest pain with breathing",
    "fatigue_weakness": "Fatigue or weakness",
    "loss_of_appetite": "Loss of appetite",
    "watery_diarrhea": "Watery diarrhea",
    "abdominal_cramps_pain": "Abdominal cramps or pain",
    "headache_body_aches": "Headache or body aches",
    "dehydration_symptoms": "Dehydration symptoms",
    "persistent_cough_3_weeks": "Persistent cough (3 weeks)",
    "coughing_up_blood": "Coughing up blood",
    "night_sweats": "Night sweats",
    "fever_chills": "Fever and chills",
    "unintentional_weight_loss": "Unintentional weight loss",
    "chest_pain_discomfort": "Chest pain or discomfort",
    "chronic_productive_cough": "Chronic productive cough",
    "shortness_of_breath_exertion": "Shortness of breath on exertion",
    "wheezing_noisy_breathing": "Wheezing or noisy breathing",
    "frequent_respiratory_infections": "Frequent respiratory infections",
    "fatigue_reduced_exercise_tolerance": "Fatigue or reduced exercise tolerance",
    "joint_pain_swelling_symmetrical": "Joint pain or swelling (symmetrical)",
    "morning_stiffness_30mins": "Morning stiffness (30 minutes)",
    "fatigue_fever_malaise": "Fatigue, fever, or malaise",
    "reduced_range_of_motion": "Reduced range of motion",
    "rheumatoid_nodules": "Rheumatoid nodules",
    "joint_deformities_long_term": "Joint deformities (long term)",
    "joint_pain_with_activity": "Joint pain with activity",
    "stiffness_less_30mins": "Stiffness (less than 30 minutes)",
    "reduced_flexibility": "Reduced flexibility",
    "bone_spurs": "Bone spurs",
    "mild_joint_swelling": "Mild joint swelling",
    "crepitus_joints": "Crepitus in joints",
    "fatigue_sluggishness": "Fatigue or sluggishness",
    "weight_gain": "Weight gain",
    "cold_intolerance": "Cold intolerance",
    "dry_skin_brittle_hair": "Dry skin or brittle hair",
    "constipation": "Constipation",
    "muscle_weakness_aches": "Muscle weakness or aches",
    "depression_low_mood": "Depression or low mood",
    "slow_heart_rate": "Slow heart rate",
    "extreme_fatigue": "Extreme fatigue",
    "fever": "Fever",
    "severe_sore_throat": "Severe sore throat",
    "swollen_lymph_glands": "Swollen lymph glands",
    "swollen_tonsils": "Swollen tonsils",
    "headache": "Headache",
    "rash": "Rash",
    "muscle_aches": "Muscle aches",
    "itchy_blister_like_rash": "Itchy, blister-like rash",
    "fatigue_malaise": "Fatigue or malaise",
    "rash_starts_trunk_spreads": "Rash starts on trunk and spreads",
    "stiff_neck": "Stiff neck",
    "confusion_concentration_issues": "Confusion or concentration issues",
    "drowsiness_difficulty_waking": "Drowsiness or difficulty waking",
    "seizures_coma": "Seizures or coma",
    "common_cold": "Common Cold" # Added common_cold
}

# --- Reverse Mapping (Human-readable name -> Internal key) ---
reversed_words_map = {v: k for k, v in words_map.items()}

# --- Helper Functions ---
def get_readable_symptoms():
    return [words_map[symptom] for symptom in all_symptoms if symptom in words_map]

def to_internal(symptom_names: list[str]) -> list[str]:
    return [reversed_words_map[name] for name in symptom_names if name in reversed_words_map]

def to_readable(symptom_keys: list[str]) -> list[str]:
    return [words_map[key] for key in symptom_keys if key in words_map]
""")

**app.py** – integrates the modules and provides diagnosis functions

In [15]:
with open("app.py", "w") as f:
    f.write("""
from inference_engine import PrologInferenceEngine

engine = PrologInferenceEngine()
def diagnose(symptoms: list[str]):
    # Assume input symptoms are already internal (e.g., "sore_throat")
    ranking = engine.rank_diseases(symptoms)
    clean_ranking = []

    if ranking:
        for disease, prob in ranking:
            clean_name = disease.replace('_', ' ').title()
            try:
                probability = int(float(prob))
            except:
                probability = 0
            clean_ranking.append((clean_name, probability))

        return {"diagnoses": clean_ranking}
    else:
        return {"message": "No match found yet. Please consult a real doctor."}
def diagnose_text(input_text: str):
    symptoms = engine.parse_input_to_symptoms(input_text)
    if not symptoms:
        return {"message": "No recognized symptoms found. Please try again."}
    ranking = engine.rank_diseases(symptoms)
    clean_ranking = []
    if ranking:
        for disease, prob in ranking:
            clean_name = disease.replace('_', ' ').title()
            try:
                probability = int(float(prob))
            except:
                probability = 0
            clean_ranking.append((clean_name, probability))

        return {"diagnoses": clean_ranking}
    else:
        return {"message": "No match found yet. Please consult a real doctor."}
""")

 Use the App
You can now use either the **symptom checklist** or **free-text input** to get predictions.

In [16]:
from app import diagnose, diagnose_text
from database import get_readable_symptoms

# Show available symptoms
symptom_options = get_readable_symptoms()
print("Available Symptoms:")
print(symptom_options[:10]) # show first 10

Available Symptoms:
['Runny nose or stuffy nose', 'Sneezing', 'Sore throat', 'Mild cough', 'Mild headache', 'Low-grade fever', 'Mild body aches or fatigue', 'High fever', 'Chills', 'Severe headache']


In [17]:
# Accept user input from a text field
import ipywidgets as widgets
from IPython.display import display, Markdown
from app import diagnose_text

text_input = widgets.Textarea(
    value='',
    placeholder='Describe your symptoms here...',
    description='Symptoms:',
    layout=widgets.Layout(width='100%', height='100px')
)

button = widgets.Button(description="Diagnose")
output = widgets.Output()

# Define what happens on button click
def on_button_click(b):
    output.clear_output()
    with output:
        result = diagnose_text(text_input.value)
        if "diagnoses" in result:
            print("Top Possible Diagnoses:")
            for disease, prob in result["diagnoses"]:
                print(f"• {disease} — {prob}% match")
        else:
            print(result["message"])

button.on_click(on_button_click)

# Display widgets
display(text_input, button, output)

Textarea(value='', description='Symptoms:', layout=Layout(height='100px', width='100%'), placeholder='Describe…

Button(description='Diagnose', style=ButtonStyle())

Output()

In [19]:
#Accept user input from checkbox
import ipywidgets as widgets
from IPython.display import display, Markdown
from app import diagnose
from database import get_readable_symptoms, to_internal

# Get the readable symptom options
symptom_options = get_readable_symptoms()

checkboxes = [widgets.Checkbox(value=False, description=sym) for sym in symptom_options]
symptom_box = widgets.VBox(checkboxes, layout=widgets.Layout(flex_flow='column', overflow='scroll', height='300px'))

output_diag = widgets.Output()

def on_checkbox_diagnose_click(b):
    output_diag.clear_output()
    with output_diag:
        selected_readable = [cb.description for cb in checkboxes if cb.value]
        selected_internal = to_internal(selected_readable) # Convert to internal keys
        result = diagnose(selected_internal)
        if "diagnoses" in result:
            print("Top Possible Diagnoses:")
            for disease, prob in result["diagnoses"]:
                print(f"• {disease} — {prob}% match")
        else:
            print(result["message"])


checkbox_button = widgets.Button(description="Diagnose")
checkbox_button.on_click(on_checkbox_diagnose_click)

display(symptom_box, checkbox_button, output_diag)

Button(description='Diagnose', style=ButtonStyle())

Output()

##  Resource Links


In [20]:
def generate_resource_links(disease_list):
    base_urls = [
        "https://www.webmd.com/search/search_results/default.aspx?query=",
        "https://www.mayoclinic.org/search/search-results?q="
    ]
    for disease in disease_list:
        print(f"\n### {disease}\n")
        for url in base_urls:
            print(f"- [{url.split('//')[1].split('/')[0]}]({url}{disease.replace(' ', '%20')})")